# End-to-End H3 Workflow and Benchmarking

This notebook outlines the newer procedures that download selected years, preprocess them, finalize `census_data.gpkg`, estimate H3 layers, and benchmark worker settings.

Full workflow cells are guarded because they can download data or require existing local GeoPackages.


In [ ]:
from pathlib import Path

import geopandas as gpd

from istat_census_data import run_h3_census_workflow


## End-to-End Workflow from Python


In [ ]:
RUN_END_TO_END_WORKFLOW = False

data_folder = Path("census_tutorial")
aoi_path = Path("aoi.gpkg")

if RUN_END_TO_END_WORKFLOW:
    aoi = gpd.read_file(aoi_path, layer="aoi")
    workflow_result = run_h3_census_workflow(
        years=[2021],
        data_folder=data_folder,
        aoi=aoi,
        indexes=["P1", "P2"],
        region_list=[15],
        workers=4,
        chunk_size=5000,
        parallel_backend="thread",
    )
    print(workflow_result.output_path)
    print(workflow_result.layers)
else:
    print("Workflow skipped. Set RUN_END_TO_END_WORKFLOW = True after preparing an AOI.")


## End-to-End Workflow from the Script File

`scripts/run_h3_census_analysis.py` is an editable example script. Set module-level values such as `list_year`, `main_path`, `aoi_path`, `list_region`, `selected_indexes`, `boundary_ring_size`, `workers`, `chunk_size`, `parallel_backend`, `intermediate_storage`, and `intermediate_dir`, then run the script from the project folder.

Use `selected_indexes = []` when you want all additive catalogue indexes. Set `run_download`, `run_preprocess`, or `run_finalize` to `False` when those files already exist locally.


## Benchmark Worker Settings

Benchmarking should use finalized local data. Configure `scripts/benchmark_h3_estimation.py` by editing its module-level variables, not by passing CLI arguments.

```python
selected_year = 2021
selected_indexes = []
boundary_ring_size = DEFAULT_BOUNDARY_RING_SIZE
worker_counts = [1, 2, 4]
chunk_sizes = [DEFAULT_H3_CHUNK_SIZE]
parallel_backend = "thread"
intermediate_storage = "parquet"
intermediate_dir = main_path / "h3_intermediates"
keep_intermediates = True
reuse_intermediates = False
runs = 1
```

`worker_counts = [1]` runs only the serial path. Include values greater than 1 to compare parallel runs. Keep `selected_indexes = []` to benchmark all additive catalogue indexes. Keep `boundary_ring_size = DEFAULT_BOUNDARY_RING_SIZE` unless you are studying a specific AOI edge case. Set `reuse_intermediates = True` only for a later warm run that should reuse retained compatible chunk files. The benchmark writes JSON after each completed run, so long executions have progress output before the full benchmark matrix finishes.


## Optional Dask Backend

Install the optional extra before using Dask:

```bash
pip install "istat-census-data[dask]"
```

Then set `parallel_backend="dask"` in Python or `parallel_backend = "dask"` in the scripts. Dask uses the same Parquet-backed chunk files as the standard backends; `pyarrow` is already included in the base package.
